# CIL Monocular Depth Estimation
Runs on **Google Colab** (downloads data via Kaggle API) or **ETH student cluster** (scans for existing data).

## 0 — Config (edit this cell)

In [ ]:
COMPETITION_SLUG = "ethz-cil-monocular-depth-estimation-2026"   # <-- set this
DECODER_TYPE     = "transformer"             # 'transformer' or 'conv'
PRETRAINED       = True                      # ImageNet init vs random
EPOCHS           = 50
BATCH_SIZE       = 8
LR               = 1e-4

## 1 — Environment detection

In [ ]:
import sys, os

IN_COLAB   = 'google.colab' in sys.modules
ON_CLUSTER = not IN_COLAB

print(f"Environment: {'Google Colab' if IN_COLAB else 'Cluster / local'}")

## 2 — Install dependencies

In [ ]:
!git clone 

In [ ]:
!pip install -r requirements.txt

## 3 — Repo setup
On Colab: either upload this repo as a zip or clone from GitHub.  
On cluster: set `REPO_ROOT` to where you cloned the repo.

In [ ]:
if IN_COLAB:
    # Option A: clone from GitHub (if you pushed the repo)
    # !git clone https://github.com/YOUR/REPO.git /content/cil
    # REPO_ROOT = '/content/cil'

    # Option B: upload zip
    import os
    if not os.path.exists('/content/cil'):
        from google.colab import files
        print('Upload the repo zip (the CIL folder zipped):')
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        !unzip -q "{fname}" -d /content/cil_unzipped
        # find the actual root (handles nested zip dirs)
        import glob
        candidates = glob.glob('/content/cil_unzipped/**/src', recursive=True)
        REPO_ROOT = os.path.dirname(candidates[0]) if candidates else '/content/cil_unzipped'
    else:
        REPO_ROOT = '/content/cil'
else:
    # On cluster: notebook lives inside the repo, so root = two levels up from here
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'REPO_ROOT={REPO_ROOT}')

## 4 — Google Drive (Colab only)
Mount Drive so checkpoints survive session restarts.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/cil_checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f'Checkpoints → {CHECKPOINT_DIR}')
else:
    CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'checkpoints')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f'Checkpoints → {CHECKPOINT_DIR}')

## 5 — Data setup
**Colab**: downloads from Kaggle. Credentials via Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) or manual upload of `kaggle.json`.  
**Cluster**: scans common HPC paths for existing data.

In [ ]:
import glob as _glob

def _find_depth_data():
    """Scan common cluster paths for train images + depth maps."""
    username = os.environ.get('USER', os.environ.get('USERNAME', 'user'))
    scratch  = os.environ.get('SCRATCH', '')
    candidates = [
        os.path.join(REPO_ROOT, 'data'),
        os.path.join(REPO_ROOT, '..', 'data'),
        os.path.expanduser('~/cil_data'),
        os.path.expanduser('~/data/cil'),
        f'/cluster/scratch/{username}/cil_data',
        f'/scratch/{username}/cil_data',
        os.path.join(scratch, 'cil_data') if scratch else '',
    ]
    for root in candidates:
        if not root:
            continue
        root = os.path.normpath(root)
        # accept any layout that has train images and at least one .npy depth file
        imgs  = _glob.glob(os.path.join(root, '**', '*.png'), recursive=True)
        npys  = _glob.glob(os.path.join(root, '**', '*.npy'), recursive=True)
        if imgs and npys:
            return root
    return None


if IN_COLAB:
    DATA_ROOT = '/content/data'
    os.makedirs(DATA_ROOT, exist_ok=True)

    if not _glob.glob(os.path.join(DATA_ROOT, '**', '*.npy'), recursive=True):
        
        # 1. Try to get credentials from different sources
        kaggle_user = None
        kaggle_key = None

        # Attempt A: Colab Secrets (Web UI only)
        try:
            from google.colab import userdata
            kaggle_user = userdata.get('KAGGLE_USERNAME')
            kaggle_key = userdata.get('KAGGLE_KEY')
        except Exception:
            pass

        # Attempt B: .env file or Environment Variables (VS Code / Local)
        if not kaggle_user or not kaggle_key:
            # Requires: pip install python-dotenv
            try:
                from dotenv import load_dotenv
                load_dotenv() 
            except ImportError:
                pass
            
            kaggle_user = os.getenv('KAGGLE_USERNAME')
            kaggle_key = os.getenv('KAGGLE_KEY')

        # 2. Write the Kaggle JSON if credentials were found
        if kaggle_user and kaggle_key:
            creds = {"username": kaggle_user, "key": kaggle_key}
            kaggle_path = os.path.expanduser('~/.kaggle')
            os.makedirs(kaggle_path, exist_ok=True)
            
            with open(os.path.join(kaggle_path, 'kaggle.json'), 'w') as f:
                json.dump(creds, f)
            
            os.chmod(os.path.join(kaggle_path, 'kaggle.json'), 0o600)
            print('Kaggle credentials configured.')
        else:
            print('Credentials not found in Secrets or .env. Manual upload required.')
else:
    found = _find_depth_data()
    if found:
        DATA_ROOT = found
        print(f'Found data at {DATA_ROOT}')
    else:
        DATA_ROOT = os.path.join(REPO_ROOT, 'data')
        print(f'Data not found. Set DATA_ROOT manually or download to {DATA_ROOT}')

print(f'DATA_ROOT={DATA_ROOT}')

## 6 — Detect data layout
Figures out where train images, train depths, and test images live inside `DATA_ROOT`.

In [ ]:
import glob as _glob

def _find_dir(root, *keywords):
    """Return first subdir whose path contains ALL keywords (case-insensitive)."""
    for dirpath, dirnames, filenames in os.walk(root):
        lower = dirpath.lower()
        if all(k in lower for k in keywords):
            imgs = _glob.glob(os.path.join(dirpath, '*.png')) + _glob.glob(os.path.join(dirpath, '*.jpg'))
            npys = _glob.glob(os.path.join(dirpath, '*.npy'))
            if imgs or npys:
                return dirpath
    return None

TRAIN_IMAGE_DIR = _find_dir(DATA_ROOT, 'train') or _find_dir(DATA_ROOT, 'image')
TRAIN_DEPTH_DIR = _find_dir(DATA_ROOT, 'depth') or _find_dir(DATA_ROOT, 'npy')
TEST_IMAGE_DIR  = _find_dir(DATA_ROOT, 'test')  or _find_dir(DATA_ROOT, 'val')

print(f'train images : {TRAIN_IMAGE_DIR}')
print(f'train depths : {TRAIN_DEPTH_DIR}')
print(f'test  images : {TEST_IMAGE_DIR}')

# quick sanity check
n_imgs  = len(_glob.glob(os.path.join(TRAIN_IMAGE_DIR, '*.png')) + _glob.glob(os.path.join(TRAIN_IMAGE_DIR, '*.jpg'))) if TRAIN_IMAGE_DIR else 0
n_npys  = len(_glob.glob(os.path.join(TRAIN_DEPTH_DIR, '*.npy'))) if TRAIN_DEPTH_DIR else 0
n_test  = len(_glob.glob(os.path.join(TEST_IMAGE_DIR,  '*.png')) + _glob.glob(os.path.join(TEST_IMAGE_DIR, '*.jpg'))) if TEST_IMAGE_DIR else 0
print(f'Found {n_imgs} train images, {n_npys} depth maps, {n_test} test images')

assert n_imgs > 0 and n_npys > 0, 'Train data missing — check DATA_ROOT and layout above!'

## 7 — Build config

In [ ]:
import yaml

cfg = {
    'model': {
        'encoder_pretrained': PRETRAINED,
        'decoder_type':       DECODER_TYPE,
        'decoder_blocks':     5,
        'embed_dim':          384,
        'num_heads':          6,
        'img_size':           560,
        'patch_size':         16,
    },
    'data': {
        'data_root':       DATA_ROOT,
        'train_image_dir': os.path.relpath(TRAIN_IMAGE_DIR, DATA_ROOT),
        'train_depth_dir': os.path.relpath(TRAIN_DEPTH_DIR, DATA_ROOT),
        'test_image_dir':  os.path.relpath(TEST_IMAGE_DIR,  DATA_ROOT) if TEST_IMAGE_DIR else 'test/images',
        'num_workers':     2,
        'val_split':       0.1,
    },
    'training': {
        'epochs':       EPOCHS,
        'batch_size':   BATCH_SIZE,
        'lr':           LR,
        'weight_decay': 0.01,
        'grad_clip':    1.0,
        'amp':          True,
        'seed':         42,
    },
    'logging': {
        'log_interval':    20,
        'checkpoint_dir':  CHECKPOINT_DIR,
        'experiment_name': 'vit_depth',
    },
}

cfg_path = os.path.join(REPO_ROOT, 'configs', 'runtime_config.yaml')
os.makedirs(os.path.dirname(cfg_path), exist_ok=True)
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)
print(f'Config written to {cfg_path}')
print(yaml.dump(cfg))

## 8 — Train

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, os.path.join(REPO_ROOT, 'src', 'train.py'),
    '--config', cfg_path,
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT)
print('Exit code:', result.returncode)

## 9 — Predict & download submission

In [ ]:
run_name   = f"vit_depth_{DECODER_TYPE}_pretrainedTrue"
CKPT_PATH  = os.path.join(CHECKPOINT_DIR, run_name, 'best.pth')
PRED_DIR   = os.path.join(REPO_ROOT, 'predictions') if not IN_COLAB else '/content/predictions'

assert os.path.exists(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'
assert TEST_IMAGE_DIR, 'TEST_IMAGE_DIR not set'

cmd = [
    sys.executable, os.path.join(REPO_ROOT, 'src', 'predict.py'),
    '--checkpoint', CKPT_PATH,
    '--test_dir',   TEST_IMAGE_DIR,
    '--output_dir', PRED_DIR,
    '--batch_size', '8',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT)

In [ ]:
# Run the competition's submission generation script
# !python <path-to-provided-submission-script> --pred_dir "{PRED_DIR}"

# On Colab: download the resulting CSV
if IN_COLAB:
    csv_files = _glob.glob(os.path.join(PRED_DIR, '*.csv'))
    if csv_files:
        from google.colab import files
        for f in csv_files:
            files.download(f)
    else:
        print('No CSV found — run the submission script first.')